# Get To Know A Dataset: TJ-NWP

The [TJweather Global-Regional Integrated Numerical Weather Prediction System Dataset (SD3)](https://registry.opendata.aws/tj-nwp/) provides high-resolution weather forecasts generated by a next-generation numerical weather prediction system developed by Tianji Weather based on the Super Dynamics on Cube (SD3) framework. This notebook is a guided introduction to the dataset — by the end, you will know how to access, explore, and visualize TJ-NWP data on AWS.

**Dataset highlights:**
- Three data products: Southeast Asia regional forecasts, Africa regional forecasts, and tropical cyclone track predictions
- 0.1° resolution (~12 km), hourly output, up to 10-day forecast lead time
- 149–153 meteorological, land-surface, and environmental variables in NetCDF4
- Publicly available on Amazon S3 under the [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) license

More documentation: [TJ-NWP Descriptive Documentation](https://github.com/tjweather/open-data-examples/blob/main/tj-nwp-descriptive-documentation.md)

In [ ]:
# CODING GUIDELINES
#
# This notebook requires the following Python libraries
# (install with pip if needed):
#
# xarray >= 2022.1
# h5netcdf >= 0.14
# matplotlib >= 3.5
# cartopy >= 0.20
# boto3 >= 1.26
# s3fs >= 2023.1
#
# The focus of this notebook is a 101-level introduction to TJ-NWP.
# All imports are gathered here so dependencies are clear upfront.

### Import Libraries

First we import all the Python libraries used throughout this notebook.

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import os

# For S3 access (public bucket, unsigned requests)
import s3fs

# For map projections
import cartopy.crs as ccrs
import cartopy.feature as cfeature

print("All libraries loaded successfully.")

### Q: How is the TJ-NWP dataset organized?

The TJ-NWP data is stored in a single Amazon S3 bucket (`s3://tj-nwp/`) with three key prefixes, each corresponding to a different data product:

```
s3://tj-nwp/
├── southeast-asia/       # 149 variables, Southeast Asia (90°E–140°E, 10°S–30°N)
├── africa/               # 153 variables, Africa (17.5°W–51.5°E, 35°S–37.5°N)
└── typhoon-track/        # 10 track & intensity parameters, Western North Pacific
```

Within each regional forecast prefix, data is organized by date and initialization cycle:

In [ ]:
# Define the S3 paths for the three TJ-NWP data products
BUCKET = "tj-nwp"

SEA_PREFIX = "southeast-asia"
AFR_PREFIX = "africa"
TC_PREFIX  = "typhoon-track"

# The data are organized as:
# s3://tj-nwp/<prefix>/YYYY/MM/YYYYMMDD/<cycle>/<filename.nc>
#
# For example, a Southeast Asia file from the 12Z cycle on 2026-05-15:
# s3://tj-nwp/southeast-asia/2026/05/20260515/12z/tj-nwp-sea.2026051512.t12z.f001.nc

print("S3 Bucket:", BUCKET)
print("Available prefixes:")
print(f"  - s3://{BUCKET}/{SEA_PREFIX}/")
print(f"  - s3://{BUCKET}/{AFR_PREFIX}/")
print(f"  - s3://{BUCKET}/{TC_PREFIX}/")

Let's connect to the TJ-NWP S3 bucket and explore its top-level structure. Since this is a public dataset, we use unsigned requests.

In [ ]:
# Create an S3 filesystem with anonymous access (the bucket is public)
fs = s3fs.S3FileSystem(anon=True)

# List the top-level prefixes in the bucket
print("Top-level prefixes in s3://tj-nwp/:")
for prefix in fs.ls("tj-nwp"):
    print(f"  {prefix}")
print()

# Peek at a date subdirectory under the Southeast Asia prefix
print("Date directories under s3://tj-nwp/southeast-asia/:")
sample = fs.ls("tj-nwp/southeast-asia/2026/05/")
for item in sample:
    print(f"  {item}")

### Q: What data formats are present in this dataset?

All TJ-NWP data are stored in **NetCDF4** format (`.nc` files), the standard format for multidimensional scientific data in the atmospheric sciences.

**Why NetCDF4?**
- **Self-describing**: each file contains its own variable names, units, and coordinate information
- **Multidimensional**: natively handles data with dimensions like (time × latitude × longitude × pressure level)
- **Efficient I/O**: supports lazy loading — you can open a 75+ MB file and only read the variable/region you need
- **Ecosystem**: fully supported by `xarray`, the standard Python library for working with labeled multidimensional arrays

**Recommended tools:**
| Tool | Purpose |
|------|---------|
| `xarray` + `h5netcdf` | Open, query, and analyze NetCDF data |
| `s3fs` | Access S3 objects as a POSIX-like filesystem |
| `cartopy` | Plot georeferenced field maps |
| `matplotlib` | General-purpose visualization |

**AWS services** that work well with this dataset:
- [Amazon Athena](https://docs.aws.amazon.com/athena/) — SQL queries over S3 (via NetCDF-compatible connectors)
- [Amazon SageMaker](https://aws.amazon.com/sagemaker/) — Notebook-based analysis and ML training
- [EC2 + EBS](https://aws.amazon.com/ec2/) — High-throughput compute with fast local storage

### Q: Can you show us an example of loading data?

Let's load a sample forecast file from each data product. We'll work with local copies first (these files are also accessible directly from S3 via `s3://` URLs).

Each regional forecast file corresponds to **one forecast hour** from a specific initialization time. The dimensions are `time × lat × lon`, where the time dimension has size 1 for a single forecast lead time.

#### Southeast Asia Regional Forecast

In [ ]:
# Load the Southeast Asia sample file (forecast hour 001 from 2026-05-15 12Z)

ds_sea = xr.open_dataset("s3://tj-nwp/southeast-asia/2026/05/20260515/12z/tj-nwp-sea.2026051512.t12z.f001.nc", engine="h5netcdf", storage_options={"anon": True})

# Show dataset summary
print("=== Southeast Asia Dataset ===")
print(ds_sea)
print()

# Key dimensions
print(f"Latitude range:  {float(ds_sea.lat.min()):.1f} to {float(ds_sea.lat.max()):.1f} °N")
print(f"Longitude range: {float(ds_sea.lon.min()):.1f} to {float(ds_sea.lon.max()):.1f} °E")
print(f"Grid size:       {ds_sea.sizes['lat']} × {ds_sea.sizes['lon']}")
print(f"Number of variables: {len(ds_sea.data_vars)}")
print(f"File size:       ~75 MB per forecast hour")

Let's look at a few key surface variables to understand the data structure.

In [ ]:
# Select and display a few key variables
sample_vars = ds_sea[["t2mz", "rh2m", "UGRD10m", "VGRD10m", "PRATEsfc", "slp"]]
print("Sample surface variables:")
for vn in list(sample_vars.data_vars):
    v = sample_vars[vn]
    print(f"  {vn:15s} | {v.long_name[:50]:50s} | units: {v.units}")
print()

# Show a small spatial slice of 2m temperature
print("2m temperature (2°×2° sub-region sample):")
t2m_sample = ds_sea["t2mz"].sel(lat=slice(15, 17), lon=slice(110, 112))
print(t2m_sample.values)

#### Africa Regional Forecast

In [ ]:
# Load the Africa sample file
ds_afr = xr.open_dataset("s3://tj-nwp/africa/2026/05/20260515/12z/tj-nwp-afr.2026051512.t12z.f001.nc", engine="h5netcdf", storage_options={"anon": True})

print("=== Africa Dataset ===")
print(ds_afr)
print()
print(f"Latitude range:  {float(ds_afr.lat.min()):.1f} to {float(ds_afr.lat.max()):.1f} °N")
print(f"Longitude range: {float(ds_afr.lon.min()):.1f} to {float(ds_afr.lon.max()):.1f} °E")
print(f"Grid size:       {ds_afr.sizes['lat']} × {ds_afr.sizes['lon']}")
print(f"Number of variables: {len(ds_afr.data_vars)} (includes 4 dust variables)")
print(f"File size:       ~187 MB per forecast hour")

# Show dust-specific variables (unique to Africa)
dust_vars = [v for v in ds_afr.data_vars if "dust" in v or v == "dod"]
print(f"\nDust-specific variables: {dust_vars}")

#### Typhoon Track Data

In [ ]:
# Load the typhoon track sample file
ds_tc = xr.open_dataset("s3://tj-nwp/typhoon-track/2026/05/20260510/12z/tc_WNP_HAGUPIT_track.nc", engine="h5netcdf", storage_options={"anon": True})

print("=== Typhoon Track Dataset ===")
print(ds_tc)
print()

# Show key track parameters
print("Typhoon track variables:")
for vn in list(ds_tc.data_vars):
    v = ds_tc[vn]
    desc = v.long_name if hasattr(v, 'long_name') else ''
    unit = v.units if hasattr(v, 'units') else ''
    print(f"  {vn:8s} | {desc[:45]:45s} | {unit}")
print()
print(f"Forecast steps: {ds_tc.sizes['time']}")
print(f"Number of ensemble members: {int(ds_tc['nmember'].values[0])}")

### Q: A picture is worth a thousand words — show us a visual!

Now let's create several visualizations that demonstrate the richness of the TJ-NWP dataset. We'll plot:

1. **Southeast Asia 2m temperature** — a full-domain spatial map
2. **Southeast Asia 10m wind field** — wind barbs overlaid on sea-level pressure
3. **Typhoon track** — the predicted storm path with intensity

#### 1. Southeast Asia 2m Temperature Map

In [ ]:
# Plot 2m temperature over Southeast Asia domain
ds = ds_sea  # alias

fig = plt.figure(figsize=(14, 8))
ax = plt.axes(projection=ccrs.PlateCarree())

# Add map features
ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.3)
ax.add_feature(cfeature.OCEAN, facecolor='#E6F0FA', zorder=0)
ax.add_feature(cfeature.LAND, facecolor='#F5F0E8', zorder=0)

# Plot temperature (squeeze the singleton time dimension)
temp = ds["t2mz"].squeeze() - 273.15  # Convert K to °C
im = temp.plot.pcolormesh(
    ax=ax, cmap='RdYlBu_r', transform=ccrs.PlateCarree(),
    add_colorbar=True, cbar_kwargs={'label': '2m Temperature (°C)', 'shrink': 0.75}
)

# Format axes
ax.set_xticks(np.arange(90, 145, 10))
ax.set_yticks(np.arange(-10, 35, 5))
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%d°E'))
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%d°N'))
gl = ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.5)

ax.set_title('TJ-NWP Southeast Asia 2m Temperature\n2026-05-15 12Z, Forecast Hour +001', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

#### 2. 10m Wind Field + Sea-Level Pressure

In [ ]:
# Plot 10m wind barbs over sea-level pressure
fig = plt.figure(figsize=(14, 8))
ax = plt.axes(projection=ccrs.PlateCarree())

# Map features
ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.3)
ax.add_feature(cfeature.OCEAN, facecolor='#E6F0FA', zorder=0)
ax.add_feature(cfeature.LAND, facecolor='#F5F0E8', zorder=0)

# Subsample grid for wind barbs (every 20th grid point)
skip = 20
u10 = ds["UGRD10m"].squeeze()
v10 = ds["VGRD10m"].squeeze()
slp_data = ds["slp"].squeeze()

# Plot SLP contours with labels
slp_levels = np.arange(float(slp_data.min()) - 2, float(slp_data.max()) + 2, 2)
cs = slp_data.plot.contour(ax=ax, levels=slp_levels, colors='black', linewidths=0.8,
                           transform=ccrs.PlateCarree())
ax.clabel(cs, cs.levels, inline=True, fontsize=7, fmt='%.0f')

# Plot wind barbs
lon, lat = np.meshgrid(ds.lon.values, ds.lat.values)
ax.barbs(lon[::skip, ::skip], lat[::skip, ::skip],
         u10.values[::skip, ::skip], v10.values[::skip, ::skip],
         length=6, linewidth=0.5, color='darkblue', transform=ccrs.PlateCarree())

# Format axes
ax.set_xticks(np.arange(90, 145, 10))
ax.set_yticks(np.arange(-10, 35, 5))
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%d°E'))
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%d°N'))

ax.set_title('TJ-NWP 10m Wind + Sea-Level Pressure\n2026-05-15 12Z, Forecast Hour +001',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

#### 3. Typhoon Track — Super Typhoon Hagupit

In [ ]:
# Plot the forecasted typhoon track with intensity shading
fig = plt.figure(figsize=(10, 9))
ax = plt.axes(projection=ccrs.PlateCarree())

# Map features
ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.3)
ax.add_feature(cfeature.OCEAN, facecolor='#E6F0FA', zorder=0)
ax.add_feature(cfeature.LAND, facecolor='#F5F0E8', zorder=0)

lon = ds_tc["lon"].values
lat = ds_tc["lat"].values
vmax = ds_tc["vmax"].values
mslp = ds_tc["mslp"].values

# Plot track colored by wind speed
sc = ax.scatter(lon, lat, c=vmax, cmap='YlOrRd', s=80, edgecolors='black',
                linewidth=0.5, zorder=5, transform=ccrs.PlateCarree())
ax.plot(lon, lat, color='darkred', linewidth=1.5, alpha=0.7, 
        transform=ccrs.PlateCarree())

# Mark first and last positions
ax.scatter(lon[0], lat[0], marker='o', s=150, facecolor='green', edgecolors='black',
           zorder=6, label='Start (t=0)', transform=ccrs.PlateCarree())
ax.scatter(lon[-1], lat[-1], marker='s', s=150, facecolor='red', edgecolors='black',
           zorder=6, label=f'End (t=+{len(lon)*6-6}h)', transform=ccrs.PlateCarree())

# Colorbar
cbar = plt.colorbar(sc, ax=ax, shrink=0.7, pad=0.08)
cbar.set_label('Max Surface Wind Speed (m/s)', fontsize=11)

# Set map extent
buffer = 5
ax.set_extent([lon.min() - buffer, lon.max() + buffer,
               lat.min() - buffer, lat.max() + buffer])

ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.5,
             xlocs=np.arange(110, 155, 5), ylocs=np.arange(5, 45, 5))
ax.legend(loc='upper left')
ax.set_title('TJ-NWP Typhoon Track Forecast\nSuper Typhoon Hagupit | Init: 2026-05-06 18Z',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Print summary statistics
print("Track Summary:")
print(f"  Start: ({lon[0]:.1f}°E, {lat[0]:.1f}°N), Max wind: {vmax[0]:.1f} m/s")
print(f"  Peak:  ({lon[vmax.argmax()]:.1f}°E, {lat[vmax.argmax()]:.1f}°N), "
      f"Max wind: {vmax.max():.1f} m/s, Min SLP: {mslp.min():.0f} hPa")
print(f"  End:   ({lon[-1]:.1f}°E, {lat[-1]:.1f}°N), Max wind: {vmax[-1]:.1f} m/s")
print(f"  Total forecast length: {len(lon) * 6} hours")

### Q: What is one question that you have answered using these data?

**How does the forecasted typhoon intensity evolve over time, and when does the storm reach its peak?**

Using the TJ-NWP typhoon track dataset, we can trace the full lifecycle of a tropical cyclone. The track data provides the minimum sea-level pressure (mslp) and maximum surface wind speed (vmax) at every 6-hour forecast step. From these, we can identify the rapid intensification phase, peak intensity, and subsequent weakening — information critical for operational routing and marine safety.

Below we plot the intensity evolution of Super Typhoon Hagupit as forecast by TJ-NWP.

In [ ]:
# Intensity evolution plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

fcst_hours = np.arange(0, len(vmax) * 6, 6)

# Max wind speed
ax1.plot(fcst_hours, vmax, 'o-', color='darkred', linewidth=2, markersize=6)
ax1.fill_between(fcst_hours, 0, vmax, alpha=0.15, color='darkred')
ax1.axhline(y=33, color='orange', linestyle='--', linewidth=1, alpha=0.7, label='Typhoon (>33 m/s)')
ax1.axhline(y=51, color='purple', linestyle='--', linewidth=1, alpha=0.7, label='Super Typhoon (>51 m/s)')
ax1.set_xlabel('Forecast Hour')
ax1.set_ylabel('Max Surface Wind Speed (m/s)')
ax1.set_title('Wind Speed Evolution')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# Minimum SLP
ax2.plot(fcst_hours, mslp, 's-', color='darkblue', linewidth=2, markersize=6)
ax2.fill_between(fcst_hours, mslp.max(), mslp, alpha=0.15, color='darkblue')
ax2.set_xlabel('Forecast Hour')
ax2.set_ylabel('Minimum Sea-Level Pressure (hPa)')
ax2.set_title('Central Pressure Evolution')
ax2.invert_yaxis()  # Lower pressure = stronger storm
ax2.grid(True, alpha=0.3)

fig.suptitle('TJ-NWP Super Typhoon Hagupit — Intensity Evolution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Identify key phases
peak_idx = vmax.argmax()
print(f"Rapid intensification: Forecast +{fcst_hours[0]:.0f}h → +{fcst_hours[peak_idx]:.0f}h")
print(f"Peak intensity:        +{fcst_hours[peak_idx]:.0f}h, {vmax[peak_idx]:.1f} m/s, {mslp[peak_idx]:.0f} hPa")
print(f"Weakening phase:       +{fcst_hours[peak_idx]:.0f}h → +{fcst_hours[-1]:.0f}h")

### Q: What is one unanswered question you think the community could tackle?

**Can we use TJ-NWP's Africa dust forecasts, combined with satellite observations, to predict air-quality impacts over populated regions?**

The Africa dataset uniquely includes four dust-specific variables (`dod`, `dust_conc`, `dust_ddep`, `dust_emis`) that describe the full lifecycle of Saharan dust — from emission through atmospheric transport to deposition. Dust storms from the Sahara regularly affect air quality across West Africa, the Mediterranean, and even the Americas.

A valuable community challenge would be to:
1. Correlate TJ-NWP dust optical depth (DOD) forecasts with satellite retrievals (e.g., MODIS, Sentinel-5P)
2. Map forecasted dust concentration to PM₁₀/PM₂.₅ estimates over major population centers
3. Develop a dust-early-warning index using TJ-NWP ensemble spread for decision-makers

This combines numerical weather prediction, satellite validation, and public-health impact assessment — an ideal interdisciplinary challenge for the open-data community.